# MangGO — Train & Export Mango Defect Detector (Colab)

Trains YOLO11n on the Roboflow mango defect dataset and exports a Core ML model
ready for the iOS app.

**Before running:** Runtime → Change runtime type → Hardware accelerator →
**T4 GPU** → Save.

Forgetting that step is the difference between 20 minutes and several hours.

In [ ]:
!nvidia-smi

: 

In [ ]:
%pip install -q ultralytics roboflow

## 1. Download the dataset

Two values to fill in:

- `API_KEY` — <https://app.roboflow.com/settings/api>
- `VERSION` — on the dataset page click **Download Dataset** → **Show download
  code**, and copy the number inside `version(...)`

In [ ]:
from roboflow import Roboflow

API_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY"
WORKSPACE = "mangotest"
PROJECT = "mango-defect-detectionv4"
VERSION = 15

dataset = (
    Roboflow(api_key=API_KEY)
    .workspace(WORKSPACE)
    .project(PROJECT)
    .version(VERSION)
    .download("yolov11")
)

DATA_YAML = f"{dataset.location}/data.yaml"
print(DATA_YAML)

In [ ]:
print(open(DATA_YAML).read())

## 2. Train

`yolo11n` is the smallest variant — the right pick for on-device inference.

`patience` stops the run early once validation stops improving, so a small
dataset does not turn into 100 epochs of overfitting.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data=DATA_YAML,
    epochs=100,
    patience=25,
    imgsz=640,
    batch=16,
    cache=True,
    project="runs",
    name="mango-defect",
    exist_ok=True,
    seed=0,
)

## 3. Metrics

Published baseline for this dataset: mAP@50 **38.2%**, precision **43.6%**,
recall **42.0%**.

Landing near that is expected — the ceiling is the dataset (401 images, one
class), not the training setup.

In [ ]:
metrics = model.val(data=DATA_YAML)

print(f"mAP@50     {metrics.box.map50:.3f}")
print(f"mAP@50-95  {metrics.box.map:.3f}")
print(f"precision  {metrics.box.mp:.3f}")
print(f"recall     {metrics.box.mr:.3f}")

In [ ]:
import os
from IPython.display import Image, display

for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    path = f"runs/mango-defect/{name}"
    if os.path.exists(path):
        print(name)
        display(Image(filename=path, width=760))

## 4. Export to Core ML

`nms=True` embeds the non-max-suppression pipeline. Without it Vision hands back
raw MultiArrays instead of `VNRecognizedObjectObservation`, and the Swift side
receives nothing.

Conversion works on Linux even though the model only *runs* on Apple hardware.
If it fails here anyway, skip ahead — the next cell still gives you `best.pt`,
and `ml/export_coreml.sh` converts it on your Mac.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = Path("runs/mango-defect/weights/best.pt")
assert WEIGHTS.exists(), f"not found: {WEIGHTS.resolve()}"

try:
    exported = Path(YOLO(str(WEIGHTS)).export(format="coreml", nms=True, imgsz=640))
    print("exported:", exported)
except Exception as error:
    exported = None
    print("Core ML export failed here:", error)
    print("Not a problem — download best.pt below and convert on your Mac.")

## 5. Download

A `.mlpackage` is a folder, not a file, so it has to be zipped before Colab can
hand it over.

In [ ]:
import shutil
from google.colab import files

if exported is not None:
    staged = Path("MangoDefect.mlpackage")
    if staged.exists():
        shutil.rmtree(staged)
    shutil.copytree(exported, staged)

    shutil.make_archive("MangoDefect", "zip", root_dir=".", base_dir="MangoDefect.mlpackage")
    files.download("MangoDefect.zip")

In [ ]:
files.download(str(WEIGHTS))

## Done — now on your Mac

1. Unzip `MangoDefect.zip`. You get a `MangoDefect.mlpackage` folder.
2. Drag it into `MangGO/MangGO/Core/Vision/Resources/`.
3. In Xcode, confirm it appears in **Build Phases → Compile Sources**.
4. Open it and check the Metadata tab reads **Object Detector**, with
   `Confidence` and `Coordinates` outputs.
5. In `iPhoneView.swift`, swap `CaptureView()` for
   `CaptureView(model: CaptureViewModel(detector: CoreMLDefectDetector()))`.
6. Build to a physical iPhone — the Simulator has no camera.

Keep the Colab tab awake while it trains. A disconnected runtime loses
everything and the run starts over.